# 05 · Max concurrency from the memory budget
Level L1 · Part 1 · core · ~25 min · builds on 03

## Goal
By the end of this lesson you can compute how many Brightside requests one GPU can run at the same time. GPU memory minus weights minus activations gives the **KV budget**. The budget divided by the memory each request needs gives the concurrency limit.

## Why it matters
- **Without it:** you guess a concurrency number, or you raise the traffic until the GPU runs out of memory. A quick guess that only divides GPU memory by request size says 101 requests on one 24 GB L4 with the 8B model. With the weights and an assumed 1 GiB for activations, the limit is about 20 (16 to 23 over 0.5 to 2 GiB), and the guess would fail under load.
- **With it:** four numbers (GPU size, weights, activations, tokens per request) and a few lines of arithmetic give a limit you can test before you buy a GPU.
- **When you don't need it:** a hosted provider sets the limits for its own GPUs and shows you a rate limit instead. You still need the math when you run the server yourself.

## The idea
A GPU's memory has three jobs. The **weights** are the model itself: fixed, loaded once. The **activations** are scratch memory for one forward step. vLLM reserves room for the worst-case step at startup, when a long prompt is prefilled. Everything left is the **KV cache** (lesson 03), and it grows with every token of every running request. Think of a car park. The building (weights) and the space set aside for the largest delivery truck (activations) are fixed. The spaces that are left are what requests park in, one space per token. A request that needs 1,800 spaces uses 1,800 of them until it finishes.

## Picture
```mermaid
flowchart LR
  G["GPU memory x 0.92 (utilisation)"] --> W["minus weights (fixed)"]
  W --> A["minus activations (step scratch)"]
  A --> K["KV budget"]
  K -->|"divided by tokens x bytes per token"| C["requests that fit"]
  C -->|"capped by max_num_seqs"| R["requests that run"]
```

## Step by step
### Step 1: the numbers we need
The 8B config is from lesson 03, so the bytes per token (128 KiB in FP16) match that lesson. Three settings come next:
- `gpu_memory_utilization`: the fraction of GPU memory the engine may use. The default is 0.92 (vLLM docs, engine args).
- `max_num_seqs`: the most requests the engine runs in one step. The default is 128 (vLLM source, `DEFAULT_MAX_NUM_SEQS`; check with `vllm serve --help` on the version you install).
- The NVIDIA L4 has 24 GB (NVIDIA's L4 datasheet).

Activations are not in any config or spec. We start from an assumed 1 GiB and test 0.5 and 2 GiB in Step 5.

In [1]:
GIB = 2**30  # bytes in one GiB, the unit the engine logs use

def kv_bytes_per_token(layers, kv_heads, head_dim, bytes_per_value=2):
    """Lesson 03: keys and values, every layer, every KV head."""
    return 2 * layers * kv_heads * head_dim * bytes_per_value

LLAMA_8B_KV = kv_bytes_per_token(32, 8, 128)   # 131,072 bytes = 128 KiB (FP16)
LLAMA_70B_KV = kv_bytes_per_token(80, 8, 128)  # 327,680 bytes = 320 KiB (FP16)

L4_BYTES = 24e9      # NVIDIA L4 spec: 24 GB
H100_BYTES = 80e9    # H100 80 GB spec
UTIL = 0.92          # vLLM docs default gpu_memory_utilization
MAX_NUM_SEQS = 128   # vLLM source default (DEFAULT_MAX_NUM_SEQS)
ACT = 1.0 * GIB      # assumed starting value (not in any config); vLLM measures its own at startup

print(f"8B KV: {LLAMA_8B_KV / 1024:.0f} KiB per token   70B KV: {LLAMA_70B_KV / 1024:.0f} KiB per token")
print(f"L4 {L4_BYTES / 1e9:.0f} GB, H100 {H100_BYTES / 1e9:.0f} GB, utilisation {UTIL}, max_num_seqs {MAX_NUM_SEQS}")

8B KV: 128 KiB per token   70B KV: 320 KiB per token
L4 24 GB, H100 80 GB, utilisation 0.92, max_num_seqs 128


### Step 2: the weights
Weights are parameters times bytes per parameter. The Llama 3.1 8B model has 8,030,261,248 parameters, which is 8.03 billion (the exact count in the Hugging Face safetensors metadata for `meta-llama/Llama-3.1-8B`). In FP16 each parameter takes 2 bytes. The 70B model (`meta-llama/Llama-3.1-70B`) has 70,553,706,496 parameters. At 2 bytes each that is about 141 GB in FP16, and at 1 byte each it is about 70 GB in FP8.

In [2]:
PARAMS_8B = 8_030_261_248
w8_fp16 = PARAMS_8B * 2
print(f"8B weights, FP16: {w8_fp16 / GIB:.1f} GiB")
print(f"70B weights, FP16: {141e9 / GIB:.1f} GiB   FP8: {70e9 / GIB:.1f} GiB")

8B weights, FP16: 15.0 GiB
70B weights, FP16: 131.3 GiB   FP8: 65.2 GiB


### Step 3: the KV budget
The engine may use `gpu_memory_utilization` times the GPU memory. It subtracts the weights and the activations. What is left is the KV budget. If nothing is left, the model does not fit, and the function says so.

In [3]:
def kv_budget(gpu_bytes, weights, activations=ACT, util=UTIL):
    """Bytes left for the KV cache after weights and activations."""
    left = gpu_bytes * util - weights - activations
    if left <= 0:
        raise ValueError(f"weights ({weights / GIB:.1f} GiB) do not fit with activations "
                         f"in {gpu_bytes * util / GIB:.1f} GiB")
    return left

budget = kv_budget(L4_BYTES, w8_fp16)
print(f"L4 budget {L4_BYTES * UTIL / GIB:.2f} GiB - weights {w8_fp16 / GIB:.2f} GiB"
      f" - assumed activations {ACT / GIB:.2f} GiB = KV budget {budget / GIB:.2f} GiB")

L4 budget 20.56 GiB - weights 14.96 GiB - assumed activations 1.00 GiB = KV budget 4.61 GiB


**What you should see:** 24 GB is 22.35 GiB, and 22.35 × 0.92 = 20.56 GiB. That 20.56 figure is the L4 budget in GiB. Subtract the 14.96 GiB of 8B weights, and about 4.6 GiB is left for the KV cache, with the assumed 1 GiB of activations. The rest of the card is outside the engine's budget.

### Step 4: the concurrency limit
One Brightside request is a 1,600-token prompt (1,500-token policy plus the question) and a 200-token answer. That is 1,800 tokens at the end. The cache has to hold all of them. Dividing the budget by that size gives the concurrency. The engine also stops at `max_num_seqs`.

In [4]:
TOKENS = 1_600 + 200  # prompt + answer, lesson 02

def max_concurrency(gpu_bytes, weights, kv_per_token, tokens, activations=ACT, util=UTIL):
    """Requests that fit by memory, and the number that actually runs (capped by max_num_seqs)."""
    by_memory = int(kv_budget(gpu_bytes, weights, activations, util) // (kv_per_token * tokens))
    return min(by_memory, MAX_NUM_SEQS), by_memory

running, by_memory = max_concurrency(L4_BYTES, w8_fp16, LLAMA_8B_KV, TOKENS)
print(f"8B on one L4, FP16 KV, {TOKENS:,} tokens per request: {by_memory} fit in memory, {running} run")

8B on one L4, FP16 KV, 1,800 tokens per request: 20 fit in memory, 20 run


**What you should see:** about 20 requests fit, and the cap of 128 does not bind. The memory is the limit.

### Step 5: the activation guess is a range, not a fact
The config does not hold the activation size. It depends on the batch and the longest prefill step. So try a range of values. vLLM measures the real peak at startup, and the server log shows the KV size it chose.

In [5]:
for act_gib in (0.5, 1.0, 2.0):
    n, _ = max_concurrency(L4_BYTES, w8_fp16, LLAMA_8B_KV, TOKENS, activations=act_gib * GIB)
    print(f"activations {act_gib:>3} GiB -> {n} concurrent requests")

activations 0.5 GiB -> 23 concurrent requests
activations 1.0 GiB -> 20 concurrent requests
activations 2.0 GiB -> 16 concurrent requests


**What you should see:** the limit falls from 23 to 16 as activations grow from 0.5 to 2 GiB. The activation number is not in the config, so measure it before you set a limit.

### Step 6: a smaller cache doubles the room
An FP8 cache stores one byte per number, so the KV per token halves to 64 KiB. The weights do not change. The L4 has FP8 tensor cores (NVIDIA's L4 datasheet lists an FP8 Tensor Core rate), so FP8 is a real option on this card.

In [6]:
LLAMA_8B_KV_FP8 = LLAMA_8B_KV // 2
n_fp8, mem_fp8 = max_concurrency(L4_BYTES, w8_fp16, LLAMA_8B_KV_FP8, TOKENS)
print(f"8B on one L4, FP8 KV: {LLAMA_8B_KV_FP8 / 1024:.0f} KiB per token -> {mem_fp8} fit, {n_fp8} run")

8B on one L4, FP8 KV: 64 KiB per token -> 41 fit, 41 run


**What you should see:** 41 requests fit with the FP8 cache, against 20 with FP16. That is about double (20 to 41, with integer rounding and a fixed activation share).

### Step 7: a 70B model on one H100
The 70B model in FP16 needs 141 GB of weights, which is more than one 80 GB H100 can hold. The function refuses. Here the weights are FP8 too, which is a different choice from the FP8 cache in Step 6 and the Try it. There the weights stay FP16 and only the KV cache is FP8. With FP8 weights and an FP8 cache, the 70B model fits, but only a few GiB are left for requests. Lesson 19 (later) explains why a bigger model needs more GPUs.

In [7]:
try:
    max_concurrency(H100_BYTES, 141e9, LLAMA_70B_KV, TOKENS)
except ValueError as err:
    print("70B FP16 on one H100:", err)

n70, mem70 = max_concurrency(H100_BYTES, 70e9, LLAMA_70B_KV // 2, TOKENS)
left70 = kv_budget(H100_BYTES, 70e9) / GIB
print(f"70B FP8 weights + FP8 KV on one H100: {left70:.1f} GiB for KV, {mem70} fit, {n70} run")

70B FP16 on one H100: weights (131.3 GiB) do not fit with activations in 68.5 GiB
70B FP8 weights + FP8 KV on one H100: 2.4 GiB for KV, 8 fit, 8 run


**What you should see:** the FP16 case prints the "do not fit" message. The FP8 case leaves about 2.4 GiB and fits about 8 requests. That is too few for a support desk, which is why the next parts (later) add GPUs.

### Step 8: check the formula against a published number
Databricks reports about 20 GiB of KV cache for Llama-2-70B at batch 64 with 1,024 tokens each, in FP16. The same formula should give that number.

In [8]:
published = 64 * 1024 * LLAMA_70B_KV / GIB
print(f"64 requests x 1,024 tokens of 70B FP16 KV: {published:.1f} GiB (Databricks: about 20 GiB)")
assert round(published) == 20

64 requests x 1,024 tokens of 70B FP16 KV: 20.0 GiB (Databricks: about 20 GiB)


## At work
- **Databricks** gets about 20 GiB of KV cache for 70B at batch 64 with 1,024 tokens, the same formula. Check your own config before a load test. → lesson 26 (later: Little's law turns this limit into QPS).
- **vLLM** sizes the KV pool from `gpu_memory_utilization`. The startup log prints the KV size and the maximum concurrency, so compare it with your number. → lesson 16 (later).
- **Meta** reports 40 to 50% prefix-cache hits at 90% HBM utilisation (HBM is the GPU's own memory). Prefix caching shares the policy prompt that this lesson counts once per request. → lesson 14 (later).
- **Runtime OOM** usually comes from activation spikes (a long prefill, a large `n`), not from the pool set aside at startup. → lesson 33 (later).

Sources: vLLM docs, engine args (`gpu_memory_utilization` default 0.92): https://docs.vllm.ai/en/latest/configuration/engine_args.html · vLLM source, `DEFAULT_MAX_NUM_SEQS` = 128 (`vllm/config/scheduler.py`): https://github.com/vllm-project/vllm · NVIDIA L4 datasheet (24 GB, FP8 Tensor Cores): https://www.nvidia.com/en-us/data-center/l4/ · Hugging Face safetensors metadata, 8,030,261,248 parameters: https://huggingface.co/api/models/meta-llama/Llama-3.1-8B · and 70,553,706,496 parameters: https://huggingface.co/api/models/meta-llama/Llama-3.1-70B · Databricks, "LLM Inference Performance Engineering: Best Practices": https://www.databricks.com/blog/llm-inference-performance-engineering-best-practices · PyTorch, "Disaggregated Inference at Scale with PyTorch & vLLM": https://pytorch.org/blog/disaggregated-inference-at-scale-with-pytorch-vllm/ · vLLM docs: https://docs.vllm.ai/

## Common mistakes
**1. Dividing the whole GPU.** This is the 101-request guess from "Why it matters". It ignores the weights, the activations and the utilisation cap, so it is too high.

In [9]:
naive = int(L4_BYTES // (LLAMA_8B_KV * TOKENS))
print(f"naive: {naive} requests (whole GPU, no weights)   budget-based: {max_concurrency(L4_BYTES, w8_fp16, LLAMA_8B_KV, TOKENS)[1]}")

naive: 101 requests (whole GPU, no weights)   budget-based: 20


**2. Counting only the prompt.** The answer tokens also sit in the cache until the request ends. Leaving them out overstates the limit.

In [10]:
print(f"prompt only (1,600 tokens): {max_concurrency(L4_BYTES, w8_fp16, LLAMA_8B_KV, 1_600)[1]} requests"
      f" vs prompt + answer (1,800): {max_concurrency(L4_BYTES, w8_fp16, LLAMA_8B_KV, TOKENS)[1]}")

prompt only (1,600 tokens): 23 requests vs prompt + answer (1,800): 20


**3. Trusting a model that does not fit.** The function raises an error when the weights leave no room. Do not catch it and go on with a made-up number. The fix is more GPUs, a smaller model or fewer bytes per parameter (lesson 19, later).

In [11]:
max_concurrency(H100_BYTES, 141e9, LLAMA_70B_KV, TOKENS)  # 70B FP16 on one H100: ValueError

ValueError: weights (131.3 GiB) do not fit with activations in 68.5 GiB

## Try it
Brightside's policy grows, and a request is now 4,000 tokens (prompt plus answer). On one L4 with the 8B model and an FP8 cache, how many requests run at once? Use `max_concurrency`.

Keep the weights in FP16 and change only the KV cache to FP8: pass `w8_fp16` unchanged as the weights, and `LLAMA_8B_KV // 2` as the KV size per token.

In [12]:
# your code here

**Answer** — try it first, then expand the hidden cell below.

In [13]:
n, mem = max_concurrency(L4_BYTES, w8_fp16, LLAMA_8B_KV // 2, 4_000)
print(f"4,000 tokens per request, FP8 KV on one L4: {mem} fit, {n} run")

4,000 tokens per request, FP8 KV on one L4: 18 fit, 18 run


## Recap
- KV budget = GPU memory × utilisation − weights − activations.
- Concurrency = KV budget ÷ (tokens per request × KV bytes per token), capped by `max_num_seqs`.
- On one 24 GB L4 with the 8B model, Brightside fits about 20 requests in FP16, and about 41 with an FP8 cache (with 1 GiB of assumed activations; vLLM's startup log gives the measured value).
- A 70B model in FP16 does not fit one 80 GB H100. It needs more GPUs (lesson 19, later).

**Interview:** "How many concurrent requests can a GPU serve?" Strong answer: subtract weights and activations from the memory the engine may use, divide by the KV bytes per request (tokens × bytes per token from the config), and cap by `max_num_seqs`. Then measure the activations, because they are the part you cannot read from the config.